# Energy Forecasting
## UK (London, Bristol, Leeds)

### 1.Daten einlesen und bearbeiten

In [31]:
# Bibliotheken importieren
import pandas as pd
import openpyxl

#### Sunshine Duration Dataframe anpassen

In [32]:
# Daten einlesen Tabellenblatt "Sunshine Duration"
energy_data_sunshine_duration = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Sunshine Duration')
energy_data_sunshine_duration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Unnamed: 0                   1524 non-null   datetime64[ns]
 1   Sunshine duration
(minutes)  1523 non-null   object        
 2   Unnamed: 2                   1523 non-null   object        
 3   Unnamed: 3                   1523 non-null   object        
 4   Unnamed: 4                   29 non-null     object        
 5   Unnamed: 5                   28 non-null     object        
 6   Unnamed: 6                   28 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 83.5+ KB


In [33]:
energy_data_sunshine_duration.rename(columns={'Unnamed: 0':'Date'}, inplace=True)
energy_data_sunshine_duration.head()

,Date,Sunshine duration\n(minutes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaT,London,Bristol,Leeds,NaN,NaN,NaN
1,2019-01-01,3.869503,213.829788,398.4,NaN,NaN,NaN
2,2019-01-02,354.398937,459.626433,457.966666,NaN,NaN,NaN
3,2019-01-03,358.707094,316.008867,358.601773,NaN,NaN,NaN
4,2019-01-04,327.997512,314.861526,112.468083,NaN,NaN,NaN


In [34]:
energy_data_sunshine_duration["Date"] = pd.to_datetime(energy_data_sunshine_duration["Date"])
energy_data_sunshine_duration.head()

,Date,Sunshine duration\n(minutes),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaT,London,Bristol,Leeds,NaN,NaN,NaN
1,2019-01-01,3.869503,213.829788,398.4,NaN,NaN,NaN
2,2019-01-02,354.398937,459.626433,457.966666,NaN,NaN,NaN
3,2019-01-03,358.707094,316.008867,358.601773,NaN,NaN,NaN
4,2019-01-04,327.997512,314.861526,112.468083,NaN,NaN,NaN


In [35]:
# Schritt 1: Erste Zeile entfernen
energy_data_sunshine_duration = energy_data_sunshine_duration.iloc[1:].reset_index(drop=True)

# Schritt 2: Richtige Spaltennamen setzen
energy_data_sunshine_duration.columns = ["Date", "Sunshine_London_per_min", "Sunshine_Bristol_per_min", "Sunshine_Leeds_per_min", "col5", "col6", "col7"]

# Schritt 3: Unnötige Spalten entfernen
energy_data_sunshine_duration = energy_data_sunshine_duration.drop(columns=["col5", "col6", "col7"])
energy_data_sunshine_duration.head()

# Schritt 4: Tagesdaten für jede Stadt extrahieren
london_bristol_leeds_sunshine_daily_data = energy_data_sunshine_duration[['Date', 'Sunshine_London_per_min', 'Sunshine_Bristol_per_min', 'Sunshine_Leeds_per_min']]

In [37]:
# Datensatz nochmals anzeigen und dann die fehlenden Daten berechnen
london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1524 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   object        
 2   Sunshine_Bristol_per_min  1522 non-null   object        
 3   Sunshine_Leeds_per_min    1522 non-null   object        
dtypes: datetime64[ns](1), object(3)
memory usage: 47.8+ KB


In [38]:
# Schritt 5: Sunshine-Daten in float umwandeln
london_bristol_leeds_sunshine_daily_data['Sunshine_London_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_London_per_min'], errors='coerce')
london_bristol_leeds_sunshine_daily_data['Sunshine_Bristol_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_Bristol_per_min'], errors='coerce')
london_bristol_leeds_sunshine_daily_data['Sunshine_Leeds_per_min'] = pd.to_numeric(london_bristol_leeds_sunshine_daily_data['Sunshine_Leeds_per_min'], errors='coerce')

london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1524 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 47.8 KB


In [46]:
# Schritt 6: # Letzten zwei Zeilen entfernen
london_bristol_leeds_sunshine_daily_data = london_bristol_leeds_sunshine_daily_data.iloc[:-2].reset_index(drop=True)
london_bristol_leeds_sunshine_daily_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1522 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 47.7 KB


#### Demand Dataframe anpassen

In [42]:
# Daten einlesen Tabellenblatt "Demand"
energy_data_demand = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Demand')
energy_data_demand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0           1522 non-null   datetime64[ns]
 1   Total Demand 
(MWh)  1522 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 23.9 KB


In [43]:
# Head anschauen
energy_data_demand.head()

,Unnamed: 0,Total Demand \n(MWh)
0,2019-01-01,25297.250000
1,2019-01-02,31779.833333
2,2019-01-03,33801.020833
3,2019-01-04,34128.791667
4,2019-01-05,31161.395833


In [44]:
#Schritt 1: Spalten sinnvoll benennen
energy_data_demand.columns = ["Date", "Total_Demand_MWh"]

#Schritt 2: Spalten-Werte prüfen
energy_data_demand["Date"] = pd.to_datetime(energy_data_demand["Date"], errors='coerce')
energy_data_demand["Total_Demand_MWh"] = pd.to_numeric(energy_data_demand["Total_Demand_MWh"], errors='coerce')

In [47]:
energy_data_demand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              1522 non-null   datetime64[ns]
 1   Total_Demand_MWh  1522 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 23.9 KB


#### Daten zusammenführen: Demand & Sunshine_City

In [48]:
energy_data_tmp = london_bristol_leeds_sunshine_daily_data.merge(energy_data_demand, on='Date', how='inner')
energy_data_tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      1522 non-null   datetime64[ns]
 1   Sunshine_London_per_min   1522 non-null   float64       
 2   Sunshine_Bristol_per_min  1522 non-null   float64       
 3   Sunshine_Leeds_per_min    1522 non-null   float64       
 4   Total_Demand_MWh          1522 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 59.6 KB


#### Temperature Dataframe miteinbinden

In [50]:
# Daten einlesen Tabellenblatt "Temperature"
energy_data_temperature = pd.read_excel('..\\Data\\Raw\\Energy Forecasting Data.xlsx', sheet_name='Temperature')
energy_data_temperature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6098 entries, 0 to 6097
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Unnamed: 0        6093 non-null   datetime64[ns]
 1   Temperature (C°)  6090 non-null   object        
 2   Unnamed: 2        6090 non-null   object        
 3   Unnamed: 3        6090 non-null   object        
 4   Unnamed: 4        106 non-null    object        
 5   Unnamed: 5        105 non-null    object        
 6   Unnamed: 6        105 non-null    object        
dtypes: datetime64[ns](1), object(6)
memory usage: 333.6+ KB


In [52]:
energy_data_temperature.head(15)

,Unnamed: 0,Temperature (C°),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaT,London,Bristol,Leeds,NaN,NaN,NaN
1,2019-01-01 06:00:00,5.786411,6.371772,7.311828,NaN,NaN,NaN
2,2019-01-01 12:00:00,4.799745,6.046772,6.441828,NaN,NaN,NaN
3,2019-01-01 18:00:00,9.206412,10.525105,7.543495,NaN,NaN,NaN
4,2019-01-02 00:00:00,4.688078,5.221772,2.925162,NaN,NaN,NaN
5,2019-01-02 06:00:00,1.563078,0.980105,0.868495,NaN,NaN,NaN
6,2019-01-02 12:00:00,0.601411,0.060105,0.941829,NaN,NaN,NaN
7,2019-01-02 18:00:00,5.278078,5.641772,4.050162,NaN,NaN,NaN
8,2019-01-03 00:00:00,1.951411,2.260105,1.846829,NaN,NaN,NaN
9,2019-01-03 06:00:00,1.516411,-0.016562,0.468495,NaN,NaN,NaN
